In [2]:
import pandas as pd
# 1. Load the data
data_dir = "D:/D_Drive/vamsi vs code/EDUKRON_FILES/Customer_churn_prediction_retenction/customer-churn-prediction-retention-model/Data"
print("Loading datasets...")
churn_df = pd.read_csv(f"{data_dir}/customer_churn.csv")
rfm_df = pd.read_csv(f"{data_dir}/customer_rfm.csv")
engagement_df = pd.read_csv(f"{data_dir}/customer_engagement_metrics.csv")
billing_df = pd.read_csv(f"{data_dir}/subscription_billing.csv")
orders_df = pd.read_csv(f"{data_dir}/orders.csv")
tickets_df = pd.read_csv(f"{data_dir}/support_tickets.csv")

Loading datasets...


In [6]:
# 2. Aggregate Transactional Data (Orders & Tickets)
# An individual customer can have multiple orders/tickets, so we need to group them by customer_id.
# Aggregate Orders: Total spend, average order value, and order coun
orders_agg = orders_df.groupby('customer_id').agg(
                total_order_amount = ('total_amount','sum'),
                average_order_amout = ('total_amount','mean'),
                total_orders = ('order_id','count')
)

In [7]:
orders_agg.head()

,total_order_amount,average_order_amout,total_orders
customer_id,,,
C000001,1869.20,934.60,2
C000004,1034.75,1034.75,1
C000005,1049.50,1049.50,1
C000006,1403.75,1403.75,1
C000007,2314.72,2314.72,1


In [12]:
tickets_agg = tickets_df.groupby('customer_id').agg(
    total_support_tickets=('ticket_id', 'count')
)


In [13]:
feature_table = churn_df.copy()


In [14]:
feature_table = feature_table.merge(rfm_df, on='customer_id', how='left')
feature_table = feature_table.merge(billing_df, on='customer_id', how='left', suffixes=('', '_bill'))
feature_table = feature_table.merge(engagement_df, on = 'customer_id', how = 'left',suffixes=('', '_eng'))
feature_table = feature_table.merge(orders_agg, on = 'customer_id', how = 'left')
feature_table = feature_table.merge(tickets_agg, on='customer_id', how='left')
# 4. Fill Missing Values (Customers who have no orders or tickets will have NaN)
feature_table['total_orders'] = feature_table['total_orders'].fillna(0)
feature_table['total_order_amount'] = feature_table['total_order_amount'].fillna(0)
feature_table['total_support_tickets'] = feature_table['total_support_tickets'].fillna(0)


In [16]:
feature_table.head()

,customer_id,tenure_months,contract_type,monthly_spend,days_since_last_order,support_tickets_30d,total_spend,order_count,avg_order_value,payment_method,...,product_tours_completed,wishlist_items,cart_abandonment_rate,promo_code_usage_90d,engagement_tier,measured_date,total_order_amount,average_order_amout,total_orders,total_support_tickets
0,C000001,88,biennial,369.85,284,10,29988.0,57,999.6,credit_card,...,7,8,0.20,13,dormant,2025-08-20,1869.2,934.6,2.0,0.0
1,C000001,88,biennial,369.85,284,10,29988.0,57,999.6,credit_card,...,0,25,0.41,13,low,2025-09-17,1869.2,934.6,2.0,0.0
2,C000001,88,biennial,369.85,284,10,29988.0,57,999.6,credit_card,...,7,8,0.20,13,dormant,2025-08-20,1869.2,934.6,2.0,0.0
3,C000001,88,biennial,369.85,284,10,29988.0,57,999.6,credit_card,...,0,25,0.41,13,low,2025-09-17,1869.2,934.6,2.0,0.0
4,C000001,88,biennial,369.85,284,10,29988.0,57,999.6,credit_card,...,7,8,0.20,13,dormant,2025-08-20,1869.2,934.6,2.0,0.0


In [21]:
feature_table.isnull().sum().sum()

np.int64(2492832)

In [22]:
# 1. Fill Transactional columns with 0
transaction_cols = ['total_orders', 'total_order_amount', 'average_order_amount', 'total_support_tickets']
for col in transaction_cols:
    if col in feature_table.columns:
        feature_table[col] = feature_table[col].fillna(0)

# 2. Separate remaining columns into Numeric and Categorical
numeric_cols = feature_table.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = feature_table.select_dtypes(include=['object']).columns

# 3. Fill Numeric columns with the Median (or 0)
for col in numeric_cols:
    # Option A: Fill with 0
    # feature_table[col] = feature_table[col].fillna(0)
    
    # Option B: Fill with Median (usually better for machine learning)
    median_val = feature_table[col].median()
    feature_table[col] = feature_table[col].fillna(median_val)

# 4. Fill Categorical columns with "Unknown"
for col in categorical_cols:
    feature_table[col] = feature_table[col].fillna('Unknown')

# Check if any nulls remain!
print("Remaining Null Values:")
print(feature_table.isnull().sum().sum()) # Should print 0


Remaining Null Values:
0


In [24]:
feature_table['churn_reason'].unique()

array(['service', 'none', 'competitor', 'price', 'relocation'],
      dtype=object)

In [26]:
feature_table.columns

Index(['customer_id', 'tenure_months', 'contract_type', 'monthly_spend',
       'days_since_last_order', 'support_tickets_30d', 'total_spend',
       'order_count', 'avg_order_value', 'payment_method', 'autopay',
       'num_products_owned', 'complaint_count', 'nps_score',
       'satisfaction_score', 'churned', 'churn_reason', 'predicted_churn_prob',
       'retention_offer_sent', 'offer_accepted', 'last_contact_date',
       'account_manager', 'updated_at', 'recency_days', 'frequency',
       'monetary', 'r_score', 'f_score', 'm_score', 'rfm_segment',
       'billing_id', 'plan_name', 'contract_type_bill',
       'monthly_recurring_revenue', 'annual_contract_value', 'billing_cycle',
       'payment_method_bill', 'autopay_enabled', 'paperless_billing',
       'late_payments_12m', 'price_increase_last_year', 'discount_pct',
       'tenure_months_bill', 'contract_end_date', 'payment_failures_90d',
       'billing_address_city', 'tax_exempt', 'invoice_delivery',
       'account_manager_b

In [27]:
import pandas as pd
import numpy as np

# We assume your dataframe is currently called 'feature_table'
print(f"Starting features: {feature_table.shape[1]}")

# -----------------------------------------------------------
# 1. RATIO FEATURES (Financial & Usage)
# -----------------------------------------------------------

# Avoid division by zero by using np.where
# Average money spent per order
feature_table['eng_avg_order_value'] = np.where(
    feature_table['total_orders'] > 0, 
    feature_table['total_order_amount'] / feature_table['total_orders'], 
    0
)

# Average value extracted per product owned
feature_table['eng_spend_per_product'] = np.where(
    feature_table['num_products_owned'] > 0,
    feature_table['total_spend'] / feature_table['num_products_owned'],
    0
)

# -----------------------------------------------------------
# 2. BEHAVIORAL INTENSITY FEATURES
# -----------------------------------------------------------

# How frequently are they opening support tickets relative to their tenure?
feature_table['eng_support_intensity'] = np.where(
    feature_table['tenure_months'] > 0,
    feature_table['support_tickets_30d'] / feature_table['tenure_months'],
    0
)

# Engagement relative to tenure (assuming you have a community_forum_posts or similar column)
# If they have 100 posts but have been here 100 months, that's normal (1/month). 
# If they have 100 posts in 2 months, they are highly active (50/month).
if 'community_forum_posts' in feature_table.columns:
    feature_table['eng_forum_posts_per_month'] = np.where(
        feature_table['tenure_months'] > 0,
        feature_table['community_forum_posts'] / feature_table['tenure_months'],
        0
    )

# -----------------------------------------------------------
# 3. CONVERTING CATEGORIES TO MEANINGFUL NUMBERS
# -----------------------------------------------------------

# Convert Contract Type strings into actual mathematical months
contract_mapping = {
    'month-to-month': 1,
    'annual': 12,
    'biennial': 24
}
if 'contract_type' in feature_table.columns:
    feature_table['eng_contract_length_months'] = feature_table['contract_type'].map(contract_mapping)
    # Fill any unmapped or missing contracts with 1 (assuming month-to-month by default)
    feature_table['eng_contract_length_months'] = feature_table['eng_contract_length_months'].fillna(1)


# -----------------------------------------------------------
# 4. BOOLEAN (YES/NO) FLAGS
# -----------------------------------------------------------

# Create a flag for "High Value Customer" (e.g., top 20% of spenders)
# This explicitly tells the model "Pay attention to this person!"
spend_80th_percentile = feature_table['total_spend'].quantile(0.80)
feature_table['eng_is_high_value'] = (feature_table['total_spend'] >= spend_80th_percentile).astype(int)

# Create a flag for "Recent Support Issues"
feature_table['eng_had_recent_support_ticket'] = (feature_table['support_tickets_30d'] > 0).astype(int)

print(f"Ending features: {feature_table.shape[1]}")
print("Feature Engineering Complete! New features have been prefixed with 'eng_'")


Starting features: 76
Ending features: 83
Feature Engineering Complete! New features have been prefixed with 'eng_'


In [ ]:
feature_table.to_csv("final_customer_feature_table.csv", index=False)